# IV. Temporalité

**Objectif** : Comprendre les habitudes temporelles de commande des clients (moment de la semaine et de la journée), afin d'identifier les périodes de forte affluence et les comportements de commandes répétées sur une même journée.

**Questions à traiter** :

1. Quand est-ce que les clients commandent le plus (heatmap jour × heure) ?
2. Les clients commandent-ils parfois plusieurs fois le même jour, et dans quelle proportion ?

**Impact business attendu** : 
Identifier les pics d'activité pour optimiser les ressources (stocks, personnel, capacité de livraison) aux moments clés, et détecter des comportements de commandes fractionnées pouvant révéler des frictions dans l'expérience d'achat.

## Setup

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import get_data
from src.features import enrich_dataframe

sns.set_theme(style="whitegrid", font_scale=1)
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.facecolor": "#fafafa",
    "figure.facecolor": "white",
})

COLORS = {
    "occasionnel": "#a8bfc1",
    "typique":     "#4290ba",
    "fidèle":      "#1b3a4b",
    "accent":      "#3cc494",
    "grey":        "#6b7280",
}
PAL_SEGMENTS = [COLORS["occasionnel"], COLORS["typique"], COLORS["fidèle"]]

data = get_data(raw_dir="../data/raw", processed_path="../data/processed/merged_orders.parquet")
data = enrich_dataframe(data)

print(f"Dataset: {data.shape[0]:,} lignes, {data.shape[1]} colonnes")

Chargement depuis le cache .parquet
Dataset: 10,737,546 lignes, 22 colonnes


## 1. Quand est-ce que les clients commandent le plus (heatmap jour × heure) ?

On mesure ici le nombre de **commandes passées** (pas de produits achetés) par jour de la semaine et heure de la journée. Comme `data` contient une ligne par produit dans une commande, on déduplique d'abord par `order_id` pour ne compter chaque commande qu'une seule fois, quelle que soit la taille de son panier.